In [ ]:
!pip install google-generativeai pydantic python-dotenv

In [ ]:
import google.generativeai as genai
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = ""

genai.configure(api_key=API_KEY)


In [ ]:
knowledge_base = {
  "music theory": {
    "keywords": ["music theory", "harmony", "melody", "rhythm", "composition"],
    "intent": "music_theory_definition",
    "response": {
        "definition": "Music theory is the study of how music works. It examines the language and notation of music, covering elements like pitch, rhythm, harmony, and form.",
        "algorithm": "To analyze a piece using music theory, one might identify the key, analyze harmonic progressions, examine melodic contours, and study rhythmic patterns.",
        "complexity": "The complexity of music theory can range from basic concepts like scales and chords (O(1) perception) to advanced counterpoint and fugue (O(n) analysis depending on piece length).",
        "example": "Understanding why a C major chord sounds 'resolved' after a G7 chord is a practical application of music theory.",
        "summary": "Music theory provides a framework for understanding, creating, and analyzing musical compositions."
    }
  },
  "instruments": {
    "keywords": ["instrument", "piano", "guitar", "violin", "drums", "flute"],
    "intent": "instrument_definition",
    "response": {
        "definition": "A musical instrument is a device created or adapted to make musical sounds. In principle, any object that produces sound can be considered a musical instrument.",
        "algorithm": "Learning an instrument typically involves understanding its mechanics, practicing scales and exercises, learning repertoire, and developing muscle memory and ear training.",
        "complexity": "The difficulty of mastering an instrument varies, from basic proficiency (O(1) effort for simple songs) to virtuosity (O(years) dedicated practice).",
        "example": "The piano uses hammers to strike strings, producing sound, while a guitar uses plucked strings.",
        "summary": "Instruments are tools through which musicians express themselves and create music."
    }
  }
}

knowledge_base["music history"] = {
    "keywords": ["music history", "eras", "periods", "composers", "genres"],
    "intent": "music_history_definition",
    "response": {
        "definition": "Music history is the academic study of changes in musical forms and cultures over time. It typically divides music into distinct periods such as Baroque, Classical, Romantic, and Modern.",
        "algorithm": "Studying music history involves researching primary sources, analyzing period-specific compositions, understanding socio-cultural contexts, and tracing stylistic developments.",
        "complexity": "Tracing the evolution of music across millennia involves extensive interdisciplinary research (O(epochs) of study).",
        "example": "Wolfgang Amadeus Mozart is a key figure of the Classical period, while Johann Sebastian Bach represents the Baroque era.",
        "summary": "Music history explores the evolution of musical expression and its profound connection to human culture."
    }
}


In [ ]:
from pydantic import BaseModel

class StudentQuery(BaseModel):
    query: str

class TutorResponse(BaseModel):
    topic: str
    definition: str
    matched: bool

In [ ]:
knowledge_base = {
  "music theory": {
    "keywords": ["music theory", "harmony", "melody", "rhythm", "composition"],
    "intent": "music_theory_definition",
    "response": {
        "definition": "Music theory is the study of how music works. It examines the language and notation of music, covering elements like pitch, rhythm, harmony, and form.",
        "algorithm": "To analyze a piece using music theory, one might identify the key, analyze harmonic progressions, examine melodic contours, and study rhythmic patterns.",
        "complexity": "The complexity of music theory can range from basic concepts like scales and chords (O(1) perception) to advanced counterpoint and fugue (O(n) analysis depending on piece length).",
        "example": "Understanding why a C major chord sounds 'resolved' after a G7 chord is a practical application of music theory.",
        "summary": "Music theory provides a framework for understanding, creating, and analyzing musical compositions."
    }
  },
  "instruments": {
    "keywords": ["instrument", "piano", "guitar", "violin", "drums", "flute"],
    "intent": "instrument_definition",
    "response": {
        "definition": "A musical instrument is a device created or adapted to make musical sounds. In principle, any object that produces sound can be considered a musical instrument.",
        "algorithm": "Learning an instrument typically involves understanding its mechanics, practicing scales and exercises, learning repertoire, and developing muscle memory and ear training.",
        "complexity": "The difficulty of mastering an instrument varies, from basic proficiency (O(1) effort for simple songs) to virtuosity (O(years) dedicated practice).",
        "example": "The piano uses hammers to strike strings, producing sound, while a guitar uses plucked strings.",
        "summary": "Instruments are tools through which musicians express themselves and create music."
    }
  }
}

knowledge_base["music history"] = {
    "keywords": ["music history", "eras", "periods", "composers", "genres"],
    "intent": "music_history_definition",
    "response": {
        "definition": "Music history is the academic study of changes in musical forms and cultures over time. It typically divides music into distinct periods such as Baroque, Classical, Romantic, and Modern.",
        "algorithm": "Studying music history involves researching primary sources, analyzing period-specific compositions, understanding socio-cultural contexts, and tracing stylistic developments.",
        "complexity": "Tracing the evolution of music across millennia involves extensive interdisciplinary research (O(epochs) of study).",
        "example": "Wolfgang Amadeus Mozart is a key figure of the Classical period, while Johann Sebastian Bach represents the Baroque era.",
        "summary": "Music history explores the evolution of musical expression and its profound connection to human culture."
    }
}

In [ ]:
def detect_intent(question, kb):
    question = question.lower()
    for topic, data in kb.items():
        for kw in data["keywords"]:
            if kw in question:
                return topic, data["response"]
    return None, None

In [ ]:
import re
import datetime
import time
import google.generativeai as genai

# ============================================================
# OPT 1: Prompt Optimization — shorter, stricter system prompt
# ============================================================
OPTIMIZED_SYSTEM_PROMPT = """You are a strict academic tutor.\nRules:\n- Use ONLY the given template content.\n- Do not add new facts. Do not paraphrase.\n- If no template exists, reply exactly: \"Sorry, I can answer only predefined academic questions.\"\n- Keep the answer plain text, no markdown headers."""

# ============================================================
# OPT 2: Intent Classification — what is the student asking for?
# ============================================================
INTENT_KEYWORDS = {
    "definition": ["what is", "define", "meaning of"],
    "explanation": ["explain", "describe", "how does"],
    "history": ["history", "origin", "evolution", "when was"],
    "example":    ["example", "illustrate", "show me"],
    "comparison": ["compare", "difference between", "vs", "versus"],
    "analysis":       ["analyze", "break down", "structure of"],
    "technique":      ["technique", "method", "how to"],
    "composer":       ["composer", "artist", "who wrote"],
    "genre":          ["genre", "style", "type of music"]
}

def classify_intent(question):
    q = question.lower()
    for label, kws in INTENT_KEYWORDS.items():
        if any(kw in q for kw in kws):
            return label
    return "definition"

# ============================================================
# OPT 3: Dynamic Prompt Selection — different instructions per task type
# ============================================================
DYNAMIC_PROMPTS = {
    "theory":    "Explain the musical concept clearly and concisely using the template.",
    "history": "Detail the historical context and evolution, using only the template.",
    "technique":      "Explain the steps or methods in order, using only the template.",
    "analysis":     "Explain the breakdown based on the template only."
}

def select_prompt_style(intent_label):
    if intent_label in ["technique", "analysis", "history"]:
        return DYNAMIC_PROMPTS["history"]
    return DYNAMIC_PROMPTS["theory"]

# ============================================================
# OPT 4: Clarification Agent — catch vague/ambiguous questions
# ============================================================
AMBIGUOUS_TOPICS = {
    "chord":   "Do you mean major, minor, dominant, or diminished chords?",
    "scale":   "Do you mean major, minor, pentatonic, or blues scale?",
    "form": "Do you mean sonata form, theme and variations, or rondo form?"
}

def check_ambiguous(question):
    q = question.lower()
    for key, clarification in AMBIGUOUS_TOPICS.items():
        specific_terms = ["major", "minor", "dominant", "diminished", "pentatonic", "blues", "sonata", "rondo"]
        if key in q and not any(term in q for term in specific_terms):
            return clarification
    return None

# ============================================================
# OPT 8: Adaptive Tutoring — difficulty level
# ============================================================
LEVEL_INSTRUCTIONS = {
    "beginner":     "Explain using simple musical terms, avoiding advanced jargon.",
    "intermediate": "Explain with moderate musical detail and standard terminology.",
    "advanced":     "Explain with full technical musical depth, including nuanced concepts and theory."
}

# ============================================================
# OPT 9: Context Awareness — remember last 2 questions
# ============================================================
conversation_history = []

def update_context(question):
    conversation_history.append(question)
    if len(conversation_history) > 2:
        conversation_history.pop(0)

def build_context_string():
    if not conversation_history:
        return ""
    return "Previous student questions: " + " | ".join(conversation_history)

# ============================================================
# OPT 5 & 6: Response Validation + Self-Reflection (simplified)
# ============================================================
def validate_response(text):
    issues = []
    if not text or len(text.strip()) < 10:
        issues.append("too short / empty")
    if text.strip().lower() == "none":
        issues.append("invalid content")
    return issues

# ============================================================
# OPT 10: Response Refinement — cleanup formatting
# ============================================================
def refine_response(text):
    if not text:
        return text
    text = re.sub(r'\n{3,}', '\n\n', text)
    lines = text.split('\n')
    seen, cleaned = set(), []
    for line in lines:
        key = line.strip().lower()
        if key == "" or key not in seen:
            cleaned.append(line)
            seen.add(key)
    return '\n'.join(cleaned).strip()

# ============================================================
# OPT 7: Retry Strategy — max 2 retries on bad output
# ============================================================
MAX_RETRIES = 2

# ============================================================
# MASTER FUNCTION — combines all 10 optimizations
# ============================================================
interaction_log = [] # Define interaction_log here if it's not defined elsewhere

def get_optimized_tutor_response(question, kb, level="intermediate"):
    start = time.time()

    # OPT 4 — clarification check first
    clarification = check_ambiguous(question)
    if clarification:
        elapsed = round(time.time() - start, 2)
        interaction_log.append({
            "timestamp": str(datetime.datetime.now()),
            "question": question, "topic": "clarification_needed",
            "response_time_sec": elapsed, "response": clarification
        })
        return clarification

    # OPT 9 — update short-term memory
    update_context(question)
    context_str = build_context_string()

    # base deterministic keyword match
    topic, template = detect_intent(question, kb)
    if not template:
        result = "Sorry, I can answer only predefined academic questions."
        elapsed = round(time.time() - start, 2)
        interaction_log.append({
            "timestamp": str(datetime.datetime.now()),
            "question": question, "topic": "unknown",
            "response_time_sec": elapsed, "response": result
        })
        return result

    # OPT 2 — classify intent
    intent_label = classify_intent(question)
    # OPT 3 — pick prompt style
    style_instruction = select_prompt_style(intent_label)
    # OPT 8 — adaptive difficulty
    level_instruction = LEVEL_INSTRUCTIONS.get(level, LEVEL_INSTRUCTIONS["intermediate"])

    model = genai.GenerativeModel(
        "gemini-flash-latest",
        system_instruction=OPTIMIZED_SYSTEM_PROMPT,   # OPT 1
        generation_config={"temperature": 0, "top_p": 1}
    )

    prompt = f"""Template: {template[intent_label] if intent_label in template else template['definition']}\n{context_str}\nDetected intent: {intent_label}\nStyle instruction: {style_instruction}\nDifficulty level: {level} - {level_instruction}\nStudent question: {question}\n\nRespond using ONLY the template content above, formatted clearly."""

    # OPT 7 — retry loop
    result = None
    attempts = 0
    while attempts <= MAX_RETRIES:
        response = model.generate_content(prompt)
        result = response.text
        issues = validate_response(result)          # OPT 5
        if not issues:
            break
        prompt += "\n\nSTRICT MODE: Answer must not be empty or invalid."
        attempts += 1
    # OPT 6 — self-reflection pass (simple re-check, log if still flawed)
    final_issues = validate_response(result)

    # OPT 10 — refine formatting
    result = refine_response(result)

    elapsed = round(time.time() - start, 2)
    interaction_log.append({
        "timestamp": str(datetime.datetime.now()),
        "question": question, "topic": topic,
        "response_time_sec": elapsed, "response": result,
        "intent": intent_label, "level": level,
        "flagged_issues": final_issues
    })
    return result

In [ ]:
knowledge_base = {
  "music theory": {
    "keywords": ["music theory", "harmony", "melody", "rhythm", "composition"],
    "intent": "music_theory_definition",
    "response": {
        "definition": "Music theory is the study of how music works. It examines the language and notation of music, covering elements like pitch, rhythm, harmony, and form.",
        "algorithm": "To analyze a piece using music theory, one might identify the key, analyze harmonic progressions, examine melodic contours, and study rhythmic patterns.",
        "complexity": "The complexity of music theory can range from basic concepts like scales and chords (O(1) perception) to advanced counterpoint and fugue (O(n) analysis depending on piece length).",
        "example": "Understanding why a C major chord sounds 'resolved' after a G7 chord is a practical application of music theory.",
        "summary": "Music theory provides a framework for understanding, creating, and analyzing musical compositions."
    }
  },
  "instruments": {
    "keywords": ["instrument", "piano", "guitar", "violin", "drums", "flute"],
    "intent": "instrument_definition",
    "response": {
        "definition": "A musical instrument is a device created or adapted to make musical sounds. In principle, any object that produces sound can be considered a musical instrument.",
        "algorithm": "Learning an instrument typically involves understanding its mechanics, practicing scales and exercises, learning repertoire, and developing muscle memory and ear training.",
        "complexity": "The difficulty of mastering an instrument varies, from basic proficiency (O(1) effort for simple songs) to virtuosity (O(years) dedicated practice).",
        "example": "The piano uses hammers to strike strings, producing sound, while a guitar uses plucked strings.",
        "summary": "Instruments are tools through which musicians express themselves and create music."
    }
  },
  "music history": {
    "keywords": ["music history", "eras", "periods", "composers", "genres"],
    "intent": "music_history_definition",
    "response": {
        "definition": "Music history is the academic study of changes in musical forms and cultures over time. It typically divides music into distinct periods such as Baroque, Classical, Romantic, and Modern.",
        "algorithm": "Studying music history involves researching primary sources, analyzing period-specific compositions, understanding socio-cultural contexts, and tracing stylistic developments.",
        "complexity": "Tracing the evolution of music across millennia involves extensive interdisciplinary research (O(epochs) of study).",
        "example": "Wolfgang Amadeus Mozart is a key figure of the Classical period, while Johann Sebastian Bach represents the Baroque era.",
        "summary": "Music history explores the evolution of musical expression and its profound connection to human culture."
    }
  }
}

In [ ]:
def detect_intent(question, kb):
    question = question.lower()
    for topic, data in kb.items():
        for kw in data["keywords"]:
            if kw in question:
                return topic, data["response"]
    return None, None

In [ ]:
import re
import datetime
import time
import google.generativeai as genai

# ============================================================
# OPT 1: Prompt Optimization — shorter, stricter system prompt
# ============================================================
OPTIMIZED_SYSTEM_PROMPT = """You are a strict academic tutor.\nRules:\n- Use ONLY the given template content.\n- Do not add new facts. Do not paraphrase.\n- If no template exists, reply exactly: \"Sorry, I can answer only predefined academic questions.\"\n- Keep the answer plain text, no markdown headers."""

# ============================================================
# OPT 2: Intent Classification — what is the student asking for?
# ============================================================
INTENT_KEYWORDS = {
    "definition": ["what is", "define", "meaning of"],
    "explanation": ["explain", "describe", "how does"],
    "history": ["history", "origin", "evolution", "when was"],
    "example":    ["example", "illustrate", "show me"],
    "comparison": ["compare", "difference between", "vs", "versus"],
    "analysis":       ["analyze", "break down", "structure of"],
    "technique":      ["technique", "method", "how to"],
    "composer":       ["composer", "artist", "who wrote"],
    "genre":          ["genre", "style", "type of music"]
}

def classify_intent(question):
    q = question.lower()
    for label, kws in INTENT_KEYWORDS.items():
        if any(kw in q for kw in kws):
            return label
    return "definition"

# ============================================================
# OPT 3: Dynamic Prompt Selection — different instructions per task type
# ============================================================
DYNAMIC_PROMPTS = {
    "theory":    "Explain the musical concept clearly and concisely using the template.",
    "history": "Detail the historical context and evolution, using only the template.",
    "technique":      "Explain the steps or methods in order, using only the template.",
    "analysis":     "Explain the breakdown based on the template only."
}

def select_prompt_style(intent_label):
    if intent_label in ["technique", "analysis", "history"]:
        return DYNAMIC_PROMPTS["history"]
    return DYNAMIC_PROMPTS["theory"]

# ============================================================
# OPT 4: Clarification Agent — catch vague/ambiguous questions
# ============================================================
AMBIGUOUS_TOPICS = {
    "chord":   "Do you mean major, minor, dominant, or diminished chords?",
    "scale":   "Do you mean major, minor, pentatonic, or blues scale?",
    "form": "Do you mean sonata form, theme and variations, or rondo form?"
}

def check_ambiguous(question):
    q = question.lower()
    for key, clarification in AMBIGUOUS_TOPICS.items():
        specific_terms = ["major", "minor", "dominant", "diminished", "pentatonic", "blues", "sonata", "rondo"]
        if key in q and not any(term in q for term in specific_terms):
            return clarification
    return None

# ============================================================
# OPT 8: Adaptive Tutoring — difficulty level
# ============================================================
LEVEL_INSTRUCTIONS = {
    "beginner":     "Explain using simple musical terms, avoiding advanced jargon.",
    "intermediate": "Explain with moderate musical detail and standard terminology.",
    "advanced":     "Explain with full technical musical depth, including nuanced concepts and theory."
}

# ============================================================
# OPT 9: Context Awareness — remember last 2 questions
# ============================================================
conversation_history = []

def update_context(question):
    conversation_history.append(question)
    if len(conversation_history) > 2:
        conversation_history.pop(0)

def build_context_string():
    if not conversation_history:
        return ""
    return "Previous student questions: " + " | ".join(conversation_history)

# ============================================================
# OPT 5 & 6: Response Validation + Self-Reflection (simplified)
# ============================================================
def validate_response(text):
    issues = []
    if not text or len(text.strip()) < 10:
        issues.append("too short / empty")
    if text.strip().lower() == "none":
        issues.append("invalid content")
    return issues

# ============================================================
# OPT 10: Response Refinement — cleanup formatting
# ============================================================
def refine_response(text):
    if not text:
        return text
    text = re.sub(r'\n{3,}', '\n\n', text)
    lines = text.split('\n')
    seen, cleaned = set(), []
    for line in lines:
        key = line.strip().lower()
        if key == "" or key not in seen:
            cleaned.append(line)
            seen.add(key)
    return '\n'.join(cleaned).strip()

# ============================================================
# OPT 7: Retry Strategy — max 2 retries on bad output
# ============================================================
MAX_RETRIES = 2

# ============================================================
# MASTER FUNCTION — combines all 10 optimizations
# ============================================================
interaction_log = []

def get_optimized_tutor_response(question, kb, level="intermediate"):
    start = time.time()

    # OPT 4 — clarification check first
    clarification = check_ambiguous(question)
    if clarification:
        elapsed = round(time.time() - start, 2)
        interaction_log.append({
            "timestamp": str(datetime.datetime.now()),
            "question": question, "topic": "clarification_needed",
            "response_time_sec": elapsed, "response": clarification
        })
        return clarification

    # OPT 9 — update short-term memory
    update_context(question)
    context_str = build_context_string()

    # base deterministic keyword match
    topic, template = detect_intent(question, kb)
    if not template:
        result = "Sorry, I can answer only predefined academic questions."
        elapsed = round(time.time() - start, 2)
        interaction_log.append({
            "timestamp": str(datetime.datetime.now()),
            "question": question, "topic": "unknown",
            "response_time_sec": elapsed, "response": result
        })
        return result

    # OPT 2 — classify intent
    intent_label = classify_intent(question)
    # OPT 3 — pick prompt style
    style_instruction = select_prompt_style(intent_label)
    # OPT 8 — adaptive difficulty
    level_instruction = LEVEL_INSTRUCTIONS.get(level, LEVEL_INSTRUCTIONS["intermediate"])

    model = genai.GenerativeModel(
        "gemini-flash-latest",
        system_instruction=OPTIMIZED_SYSTEM_PROMPT,   # OPT 1
        generation_config={"temperature": 0, "top_p": 1}
    )

    prompt = f"""Template: {template[intent_label] if intent_label in template else template['definition']}\n{context_str}\nDetected intent: {intent_label}\nStyle instruction: {style_instruction}\nDifficulty level: {level} - {level_instruction}\nStudent question: {question}\n\nRespond using ONLY the template content above, formatted clearly."""

    # OPT 7 — retry loop
    result = None
    attempts = 0
    while attempts <= MAX_RETRIES:
        response = model.generate_content(prompt)
        result = response.text
        issues = validate_response(result)          # OPT 5
        if not issues:
            break
        prompt += "\n\nSTRICT MODE: Answer must not be empty or invalid."
        attempts += 1
    # OPT 6 — self-reflection pass (simple re-check, log if still flawed)
    final_issues = validate_response(result)

    # OPT 10 — refine formatting
    result = refine_response(result)

    elapsed = round(time.time() - start, 2)
    interaction_log.append({
        "timestamp": str(datetime.datetime.now()),
        "question": question, "topic": topic,
        "response_time_sec": elapsed, "response": result,
        "intent": intent_label, "level": level,
        "flagged_issues": final_issues
    })
    return result

In [ ]:
def detect_intent(question, kb):
    question = question.lower()
    for topic, data in kb.items():
        for kw in data["keywords"]:
            if kw in question:
                return topic, data["response"]
    return None, None

In [ ]:
SYSTEM_PROMPT = """
You are a university tutoring assistant.
Rules:
- Answer only using the provided template.
- Do not add new information.
- Do not paraphrase.
- If no template exists, reply exactly:
"Sorry, I can answer only predefined academic questions."
"""

In [ ]:
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gemini-robotics-er-2-preview
models/gemini-2.5-computer-use-preview-10-2025
models/an

In [ ]:
def get_tutor_response(question, kb):
    topic, template = detect_intent(question, kb)
    if not template:
        return "Sorry, I can answer only predefined academic questions."

    model = genai.GenerativeModel(
        "gemini-flash-latest",
        system_instruction=SYSTEM_PROMPT,
        generation_config={"temperature": 0, "top_p": 1}
    )
    prompt = f"Template: {template}\nStudent question: {question}\nReturn the template response exactly."
    response = model.generate_content(prompt)
    return response.text

In [ ]:
print(get_optimized_tutor_response("What is music theory?", knowledge_base))
print(get_optimized_tutor_response("Explain the piano", knowledge_base))
print(get_optimized_tutor_response("Tell me about Baroque music", knowledge_base))
print(get_optimized_tutor_response("What is a major chord?", knowledge_base))

Music theory is the study of how music works. It examines the language and notation of music, covering elements like pitch, rhythm, harmony, and form.
A musical instrument is a device created or adapted to make musical sounds. In principle, any object that produces sound can be considered a musical instrument.
Sorry, I can answer only predefined academic questions.
Sorry, I can answer only predefined academic questions.


In [ ]:
import datetime
import time

interaction_log = []

def get_tutor_response(question, kb):
    start = time.time()
    topic, template = detect_intent(question, kb)
    if not template:
        result = "Sorry, I can answer only predefined academic questions."
    else:
        model = genai.GenerativeModel(
            "gemini-flash-latest",
            system_instruction=SYSTEM_PROMPT,
            generation_config={"temperature": 0, "top_p": 1}
        )
        prompt = f"Template: {template}\nStudent question: {question}\nReturn the template response exactly."
        response = model.generate_content(prompt)
        result = response.text

    elapsed = round(time.time() - start, 2)
    interaction_log.append({
        "timestamp": str(datetime.datetime.now()),
        "question": question,
        "topic": topic if topic else "unknown",
        "response_time_sec": elapsed,
        "response": result
    })
    return result

In [ ]:
for entry in interaction_log:
    print(entry)

In [ ]:
r1 = get_tutor_response("What is music theory?", knowledge_base)
r2 = get_tutor_response("What is music theory?", knowledge_base)
print(r1 == r2)   # should print True

True


**ALL 10 OPTIMIZATION PROMPTS **

In [ ]:
import re

# ============================================================
# OPT 1: Prompt Optimization — shorter, stricter system prompt
# ============================================================
OPTIMIZED_SYSTEM_PROMPT = """You are a strict academic tutor.
Rules:
- Use ONLY the given template content.
- Do not add new facts. Do not paraphrase.
- If no template exists, reply exactly: "Sorry, I can answer only predefined academic questions."
- Keep the answer plain text, no markdown headers."""

# ============================================================
# OPT 2: Intent Classification — what is the student asking for?
# ============================================================
INTENT_KEYWORDS = {
    "definition": ["what is", "define", "meaning of", "explain what is"],
    "explanation": ["explain", "describe", "how does", "tell me about", "elucidate"],
    "history": ["history", "origin", "evolution", "when was", "development of", "historical context", "era", "period"],
    "example":    ["example", "illustrate", "show me an example of", "give an instance of"],
    "comparison": ["compare", "difference between", "vs", "versus", "how is X different from Y", "contrast"],
    "analysis":       ["analyze", "break down", "structure of", "elements of", "compositional analysis", "examine"],
    "technique":      ["technique", "method", "how to", "way to play", "performance practice", "perform", "play", "steps for"],
    "composer":       ["composer", "artist", "who wrote", "biography of", "works by", "life of"],
    "genre":          ["genre", "style", "type of music", "category of music", "characteristics of", "musical form"],
    "theory":         ["music theory", "theoretical concept", "harmony", "melody", "rhythm", "counterpoint", "form in music"]
}

def classify_intent(question):
    q = question.lower()
    for label, kws in INTENT_KEYWORDS.items():
        if any(kw in q for kw in kws):
            return label
    return "definition"

# ============================================================
# OPT 3: Dynamic Prompt Selection — different instructions per task type
# ============================================================
DYNAMIC_PROMPTS = {
    "theory":    "Explain the musical concept clearly and concisely using the template.",
    "history": "Detail the historical context and evolution, using only the template.",
    "technique":      "Explain the steps or methods in order, using only the template.",
    "analysis":     "Explain the breakdown based on the template only."
}

def select_prompt_style(intent_label):
    if intent_label in ["technique", "analysis", "history"]:
        return DYNAMIC_PROMPTS["history"]
    return DYNAMIC_PROMPTS["theory"]

# ============================================================
# OPT 4: Clarification Agent — catch vague/ambiguous questions
# ============================================================
AMBIGUOUS_TOPICS = {
    "chord":   "Do you mean major, minor, dominant, or diminished chords?",
    "scale":   "Do you mean major, minor, pentatonic, or blues scale?",
    "form": "Do you mean sonata form, theme and variations, or rondo form?"
}

def check_ambiguous(question):
    q = question.lower()
    for key, clarification in AMBIGUOUS_TOPICS.items():
        specific_terms = ["major", "minor", "dominant", "diminished", "pentatonic", "blues", "sonata", "rondo"]
        if key in q and not any(term in q for term in specific_terms):
            return clarification
    return None

# ============================================================
# OPT 8: Adaptive Tutoring — difficulty level
# ============================================================
LEVEL_INSTRUCTIONS = {
    "beginner":     "Explain using simple musical terms, avoiding advanced jargon.",
    "intermediate": "Explain with moderate musical detail and standard terminology.",
    "advanced":     "Explain with full technical musical depth, including nuanced concepts and theory."
}

# ============================================================
# OPT 9: Context Awareness — remember last 2 questions
# ============================================================
conversation_history = []

def update_context(question):
    conversation_history.append(question)
    if len(conversation_history) > 2:
        conversation_history.pop(0)

def build_context_string():
    if not conversation_history:
        return ""
    return "Previous student questions: " + " | ".join(conversation_history)

# ============================================================
# OPT 5 & 6: Response Validation + Self-Reflection (simplified)
# NOTE: since your KB stores single-sentence templates (not full
# Topic/Definition/Algorithm/Complexity/Example/Summary fields),
# full field validation isn't meaningful yet. This checks length
# and non-emptiness as a stand-in — see note below the code.
# ============================================================
def validate_response(text):
    issues = []
    if not text or len(text.strip()) < 10:
        issues.append("too short / empty")
    if text.strip().lower() == "none":
        issues.append("invalid content")
    return issues

# ============================================================
# OPT 10: Response Refinement — cleanup formatting
# ============================================================
def refine_response(text):
    if not text:
        return text
    text = re.sub(r'\n{3,}', '\n\n', text)
    lines = text.split('\n')
    seen, cleaned = set(), []
    for line in lines:
        key = line.strip().lower()
        if key == "" or key not in seen:
            cleaned.append(line)
            seen.add(key)
    return '\n'.join(cleaned).strip()

# ============================================================
# OPT 7: Retry Strategy — max 2 retries on bad output
# ============================================================
MAX_RETRIES = 2

# ============================================================
# MASTER FUNCTION — combines all 10 optimizations
# ============================================================
def get_optimized_tutor_response(question, kb, level="intermediate"):
    start = time.time()

    # OPT 4 — clarification check first
    clarification = check_ambiguous(question)
    if clarification:
        elapsed = round(time.time() - start, 2)
        interaction_log.append({
            "timestamp": str(datetime.datetime.now()),
            "question": question, "topic": "clarification_needed",
            "response_time_sec": elapsed, "response": clarification
        })
        return clarification

    # OPT 9 — update short-term memory
    update_context(question)
    context_str = build_context_string()

    # base deterministic keyword match
    topic, template = detect_intent(question, kb)
    if not template:
        result = "Sorry, I can answer only predefined academic questions."
        elapsed = round(time.time() - start, 2)
        interaction_log.append({
            "timestamp": str(datetime.datetime.now()),
            "question": question, "topic": "unknown",
            "response_time_sec": elapsed, "response": result
        })
        return result

    # OPT 2 — classify intent
    intent_label = classify_intent(question)
    # OPT 3 — pick prompt style
    style_instruction = select_prompt_style(intent_label)
    # OPT 8 — adaptive difficulty
    level_instruction = LEVEL_INSTRUCTIONS.get(level, LEVEL_INSTRUCTIONS["intermediate"])

    model = genai.GenerativeModel(
        "gemini-flash-latest",
        system_instruction=OPTIMIZED_SYSTEM_PROMPT,   # OPT 1
        generation_config={"temperature": 0, "top_p": 1}
    )

    prompt = f"""Template: {template[intent_label] if intent_label in template else template['definition']}
{context_str}
Detected intent: {intent_label}
Style instruction: {style_instruction}
Difficulty level: {level} - {level_instruction}
Student question: {question}

Respond using ONLY the template content above, formatted clearly."""

    # OPT 7 — retry loop
    result = None
    attempts = 0
    while attempts <= MAX_RETRIES:
        response = model.generate_content(prompt)
        result = response.text
        issues = validate_response(result)          # OPT 5
        if not issues:
            break
        prompt += "\n\nSTRICT MODE: Answer must not be empty or invalid."
        attempts += 1
    # OPT 6 — self-reflection pass (simple re-check, log if still flawed)
    final_issues = validate_response(result)

    # OPT 10 — refine formatting
    result = refine_response(result)

    elapsed = round(time.time() - start, 2)
    interaction_log.append({
        "timestamp": str(datetime.datetime.now()),
        "question": question, "topic": topic,
        "response_time_sec": elapsed, "response": result,
        "intent": intent_label, "level": level,
        "flagged_issues": final_issues
    })
    return result

In [ ]:
print(get_optimized_tutor_response("What is a melody?", knowledge_base, level="beginner"))
print("---")
print(get_optimized_tutor_response("Explain chords", knowledge_base))   # should trigger clarification
print("---")
print(get_optimized_tutor_response("Explain harmony", knowledge_base))
print(get_optimized_tutor_response("Compare harmony with melody", knowledge_base))  # tests context awareness

Music theory is the study of how music works. It examines the language and notation of music, covering elements like pitch, rhythm, harmony, and form.
---
Do you mean major, minor, dominant, or diminished chords?
---
Music theory is the study of how music works. It examines the language and notation of music, covering elements like pitch, rhythm, harmony, and form.
Music theory is the study of how music works. It examines the language and notation of music, covering elements like pitch, rhythm, harmony, and form.


In [ ]:
print("="*70)
print("TEST SUITE: Music Tutoring Agent")
print("="*70)

# Test with a music-related question
print("\nTesting music-related question:")
response_music = get_optimized_tutor_response("What is music theory?", knowledge_base)
print("Music Theory Response:", response_music)

# Test with a non-music (data structure) question
print("\nTesting non-music (data structure) question:")
response_datastructure = get_optimized_tutor_response("What is a stack?", knowledge_base)
print("Data Structure Response:", response_datastructure)

# Test with an ambiguous music topic
print("\nTesting ambiguous music topic:")
response_ambiguous = get_optimized_tutor_response("Explain chords", knowledge_base)
print("Ambiguous Chord Response:", response_ambiguous)

# Test with a specific music topic (should not be ambiguous after clarification)
print("\nTesting specific music topic:")
response_specific = get_optimized_tutor_response("Explain major chords", knowledge_base)
print("Specific Chord Response:", response_specific)

print("\n" + "="*70)

TEST SUITE: Music Tutoring Agent

Testing music-related question:
Music Theory Response: Music theory is the study of how music works. It examines the language and notation of music, covering elements like pitch, rhythm, harmony, and form.

Testing non-music (data structure) question:
Data Structure Response: Sorry, I can answer only predefined academic questions.

Testing ambiguous music topic:
Ambiguous Chord Response: Do you mean major, minor, dominant, or diminished chords?

Testing specific music topic:
Specific Chord Response: Sorry, I can answer only predefined academic questions.

